# 03 — Spark SQL, UDFs, and I/O

SQL queries, UDFs, reading/writing Parquet/JSON, caching, explain.

## Setup

In [ ]:
import os, shutil, subprocess, sys

def _find_java():
    """Check if java is available on PATH or in JAVA_HOME."""
    if os.environ.get("JAVA_HOME"):
        java_bin = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
        if os.path.isfile(java_bin):
            return True
    return shutil.which("java") is not None

def _find_installed_jdk():
    """Look for a previously installed JDK in ~/.jdk."""
    jdk_dir = os.path.expanduser("~/.jdk")
    if os.path.exists(jdk_dir):
        for d in sorted(os.listdir(jdk_dir)):
            candidate = os.path.join(jdk_dir, d)
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "bin", "java")):
                return candidate
    return None

# Auto-install Java if not available (required by PySpark)
if not _find_java():
    prev = _find_installed_jdk()
    if prev:
        os.environ["JAVA_HOME"] = prev
        print(f"Using JAVA_HOME={prev}")
    else:
        print("Java not found. Installing JDK 17 via install-jdk...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "install-jdk"])
        import jdk
        path = jdk.install("17")
        os.environ["JAVA_HOME"] = path
        print(f"JAVA_HOME set to {path}")
else:
    print(f"Java found. JAVA_HOME={os.environ.get('JAVA_HOME', '(system)')}")

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType, DoubleType

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Module12-SQL-IO") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

emp_df = spark.read.csv(os.path.join(FIXTURES, "employees.csv"), header=True, inferSchema=True)
orders_df = spark.read.csv(os.path.join(FIXTURES, "orders.csv"), header=True, inferSchema=True)
customers_df = spark.read.csv(os.path.join(FIXTURES, "customers.csv"), header=True, inferSchema=True)
print("Data loaded.")

## 1. Spark SQL

### Register temp views and run SQL

In [ ]:
emp_df.createOrReplaceTempView("employees")
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")

result = spark.sql("""
    SELECT department, COUNT(*) as cnt, ROUND(AVG(salary), 0) as avg_sal
    FROM employees
    GROUP BY department
    ORDER BY avg_sal DESC
""")
result.show()

### Subqueries and CTEs

In [ ]:
spark.sql("""
    WITH dept_stats AS (
        SELECT department,
               AVG(salary) as avg_salary
        FROM employees
        GROUP BY department
    )
    SELECT e.name, e.department, e.salary,
           ROUND(d.avg_salary, 0) as dept_avg,
           ROUND(e.salary - d.avg_salary, 0) as diff_from_avg
    FROM employees e
    JOIN dept_stats d ON e.department = d.department
    ORDER BY diff_from_avg DESC
""").show()

### Joins in SQL

In [ ]:
spark.sql("""
    SELECT c.name as customer, o.product, o.amount, o.order_date
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.amount > 500
    ORDER BY o.amount DESC
""").show()

## 2. User Defined Functions (UDFs)

### Python UDF

In [ ]:
@F.udf(returnType=StringType())
def salary_grade(salary):
    if salary is None:
        return "Unknown"
    if salary >= 100000:
        return "A"
    elif salary >= 80000:
        return "B"
    elif salary >= 70000:
        return "C"
    return "D"

emp_df.withColumn("grade", salary_grade(F.col("salary"))) \
      .select("name", "salary", "grade").show()

### Pandas UDF (Vectorized — much faster)

In [ ]:
import pandas as pd

@F.pandas_udf(DoubleType())
def normalize_salary(salary: pd.Series) -> pd.Series:
    return (salary - salary.mean()) / salary.std()

emp_df.withColumn("salary_normalized", normalize_salary(F.col("salary"))) \
      .select("name", "salary", F.round("salary_normalized", 2).alias("normalized")) \
      .show()

### Register UDF for SQL

In [ ]:
spark.udf.register("salary_grade_sql", lambda s: "A" if s and s >= 100000 else "B" if s and s >= 80000 else "C", StringType())

spark.sql("""
    SELECT name, salary, salary_grade_sql(salary) as grade
    FROM employees
    ORDER BY salary DESC
    LIMIT 5
""").show()

## 3. Reading and Writing Data

### Parquet (columnar, compressed)

In [ ]:
import tempfile

tmpdir = tempfile.mkdtemp()

# Write
parquet_path = os.path.join(tmpdir, "employees.parquet")
emp_df.write.mode("overwrite").parquet(parquet_path)
print(f"Written to {parquet_path}")

# Read
df_parquet = spark.read.parquet(parquet_path)
df_parquet.show(3)
print(f"Count: {df_parquet.count()}")

### JSON

In [ ]:
json_path = os.path.join(tmpdir, "orders.json")
orders_df.write.mode("overwrite").json(json_path)

df_json = spark.read.json(json_path)
df_json.show(3)

### Partitioned writes

In [ ]:
part_path = os.path.join(tmpdir, "emp_by_dept")
emp_df.write.mode("overwrite").partitionBy("department").parquet(part_path)

# Read a single partition
eng_df = spark.read.parquet(os.path.join(part_path, "department=Engineering"))
print(f"Engineering employees: {eng_df.count()}")
eng_df.show()

## 4. Caching and Performance

In [ ]:
# Cache a DataFrame in memory
emp_df.cache()
emp_df.count()  # triggers caching

# Check if cached
print(f"Is cached: {emp_df.is_cached}")

# Unpersist
emp_df.unpersist()
print(f"After unpersist: {emp_df.is_cached}")

### Explain — viewing the query plan

In [ ]:
# See execution plan
result = emp_df.filter(F.col("salary") > 80000).groupBy("department").agg(F.avg("salary"))
result.explain(True)

### Broadcast join hint

In [ ]:
from pyspark.sql.functions import broadcast

# Force broadcast of small table
result = orders_df.join(broadcast(customers_df), on="customer_id")
result.explain()

## 5. Error Handling Patterns

In [ ]:
from pyspark.sql.utils import AnalysisException

# Handle missing columns gracefully
try:
    emp_df.select("nonexistent_column")
except AnalysisException as e:
    print(f"Caught AnalysisException: {e}")

# Safe column check
if "salary" in emp_df.columns:
    print("salary column exists")
else:
    print("salary column missing")

## Cleanup

In [ ]:
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
spark.stop()
print("Done.")